# 🛡️ VAHAAN — 3-Stage AI Pipeline Training

## Architecture
```
Frame → [Stage 1: Vehicle Classifier] → [Stage 2: Plate + OCR]
                    ↓ cropped ROI
             [Stage 3: Safety Violations]

[Parallel] → [Pothole Net: Road Surface]
```

## Training Strategy
| Model | Base | Reason |
|---|---|---|
| `stage1_vahaan.pt` | `best_new.pt` from HF | Already knows Indian roads — fine-tune only |
| `safety_net.pt` | `yolov8n.pt` (COCO) | COCO has `person` foundation for PPE |
| `pothole_net.pt` | `yolov8n.pt` (COCO) | Road surface — unrelated to vehicles |

## HuggingFace Repo Structure Expected
```
your-hf-repo/
├── base_models/best_new.pt        ← your existing trained model
├── stage1_dataset/                ← existing vehicle dataset
├── stage3_safety/                 ← merged safety datasets
└── pothole/                       ← india.v2i.yolov11
```

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION — Edit these before running
# ══════════════════════════════════════════════════════════════
HF_REPO    = "YOUR_HF_USERNAME/vahaan-training-hub"  # ← your HF repo
HF_TOKEN   = ""   # Add as Kaggle Secret if repo is private

# ── Stage 1: Vehicle Classifier (fine-tune from best_new.pt) ──
STAGE1_EPOCHS   = 50    # Fewer epochs — already well-trained
STAGE1_LR       = 0.001 # Lower LR to preserve existing knowledge
STAGE1_FREEZE   = 10    # Freeze first 10 backbone layers (preserves features)

# ── Stage 3: Safety Net (fresh train from COCO yolov8n) ────────
STAGE3_EPOCHS   = 100
STAGE3_LR       = 0.01
STAGE3_FREEZE   = 0     # No freezing — train the whole network

# ── Pothole Net (fresh train) ──────────────────────────────────
POTHOLE_EPOCHS  = 80

# ── Shared ─────────────────────────────────────────────────────
IMGSZ    = 640
BATCH    = 16   # Reduce to 8 if OOM
PATIENCE = 20   # Early stopping
# ══════════════════════════════════════════════════════════════

In [ ]:
# ── INSTALL ───────────────────────────────────────────────────
!pip install -q ultralytics huggingface_hub pyyaml
import os, shutil, yaml
from pathlib import Path
from ultralytics import YOLO
from huggingface_hub import snapshot_download, hf_hub_download
print('✅ Dependencies ready')

In [ ]:
# ── DOWNLOAD EVERYTHING FROM HUGGINGFACE ──────────────────────
DATA_ROOT   = Path('/kaggle/working/vahaan_data')
OUTPUT_DIR  = Path('/kaggle/working/vahaan_models')
DATA_ROOT.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'📥 Fetching from HuggingFace: {HF_REPO}')
snapshot_download(
    repo_id   = HF_REPO,
    repo_type = 'dataset',
    local_dir = str(DATA_ROOT),
    token     = HF_TOKEN or None,
)
print('✅ Download complete\n')

# Verify structure
BASE_MODEL_PATH  = DATA_ROOT / 'base_models' / 'best_new.pt'
STAGE1_YAML      = DATA_ROOT / 'stage1_dataset' / 'data.yaml'
STAGE3_YAML      = DATA_ROOT / 'stage3_safety'  / 'data.yaml'
POTHOLE_YAML     = DATA_ROOT / 'pothole'         / 'data.yaml'

checks = {
    'best_new.pt (Stage 1 base)': BASE_MODEL_PATH,
    'Stage 1 dataset yaml':       STAGE1_YAML,
    'Stage 3 safety yaml':        STAGE3_YAML,
    'Pothole yaml':               POTHOLE_YAML,
}
all_ok = True
for name, path in checks.items():
    ok = path.exists()
    print(f'  {"✅" if ok else "❌"} {name}: {path}')
    if not ok: all_ok = False

if not all_ok:
    print('\n⚠️ Some files are missing. Check your HF repo structure before continuing.')
else:
    print('\n✅ All files present — ready to train')

In [ ]:
# ── PATCH YAML PATHS TO ABSOLUTE KAGGLE PATHS ────────────────
def patch_yaml(yaml_path: Path) -> Path:
    """Rewrite train/val/test paths to absolute paths for Kaggle."""
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    dataset_dir = yaml_path.parent
    for split in ['train', 'val', 'test']:
        if split in cfg:
            p = dataset_dir / split / 'images'
            if p.exists():
                cfg[split] = str(p)
    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f)
    print(f'  ✅ Patched: {yaml_path.name}')
    return yaml_path

print('🔧 Patching YAML files...')
for yp in [STAGE1_YAML, STAGE3_YAML, POTHOLE_YAML]:
    if yp.exists():
        patch_yaml(yp)

# Show class info for each dataset
print('\n📋 Dataset class summary:')
for label, yp in [('Stage 1', STAGE1_YAML), ('Stage 3', STAGE3_YAML), ('Pothole', POTHOLE_YAML)]:
    if yp.exists():
        with open(yp) as f:
            cfg = yaml.safe_load(f)
        print(f'  [{label}] {cfg.get("nc","?")} classes: {cfg.get("names",[])}') 

In [ ]:
# ══════════════════════════════════════════════════════════════
# STAGE 1 — Fine-tune Vehicle Classifier from best_new.pt
# ══════════════════════════════════════════════════════════════
# Strategy: Load best_new.pt (already knows Indian vehicles)
# Freeze the backbone layers to preserve existing feature knowledge
# Only retrain the detection head with updated/expanded classes
# ══════════════════════════════════════════════════════════════
print('='*60)
print('  STAGE 1: Fine-tuning Vehicle Classifier')
print(f'  Base: best_new.pt  →  Freeze: {STAGE1_FREEZE} layers')
print('='*60)

s1_model = YOLO(str(BASE_MODEL_PATH))  # Start from YOUR model, not yolov8n

s1_results = s1_model.train(
    data    = str(STAGE1_YAML),
    epochs  = STAGE1_EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    name    = 'stage1_vahaan',
    lr0     = STAGE1_LR,          # Conservative LR — preserve existing knowledge
    lrf     = 0.01,
    freeze  = STAGE1_FREEZE,      # Freeze backbone — only retrain head
    patience= PATIENCE,
    device  = 0,
    augment = True,
    # Indian road conditions augmentation
    hsv_s   = 0.7,
    hsv_v   = 0.5,
    mosaic  = 1.0,
    flipud  = 0.05,
    warmup_epochs = 3,
)

s1_best = Path('runs/detect/stage1_vahaan/weights/best.pt')
s1_out  = OUTPUT_DIR / 'stage1_vahaan.pt'
if s1_best.exists():
    shutil.copy(s1_best, s1_out)
    print(f'\n✅ Stage 1 saved: {s1_out}')
    map50 = s1_results.results_dict.get('metrics/mAP50(B)', 'N/A')
    print(f'   mAP50: {map50}')
else:
    print('⚠️ Stage 1 best.pt not found')
    s1_out = BASE_MODEL_PATH  # fallback to original

In [ ]:
# ══════════════════════════════════════════════════════════════
# STAGE 3 — Train Safety Violation Detector (fresh from COCO)
# ══════════════════════════════════════════════════════════════
# Strategy: Start from yolov8n.pt (COCO weights)
# COCO has 'person' class → perfect foundation for helmet/seatbelt
# Full network training — no frozen layers
# ══════════════════════════════════════════════════════════════
print('='*60)
print('  STAGE 3: Training Safety Violation Detector')
print('  Base: yolov8n.pt (COCO)  — NO freezing')
print('='*60)

s3_model = YOLO('yolov8n.pt')   # COCO base — has person knowledge

s3_results = s3_model.train(
    data    = str(STAGE3_YAML),
    epochs  = STAGE3_EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    name    = 'safety_net',
    lr0     = STAGE3_LR,
    freeze  = STAGE3_FREEZE,
    patience= PATIENCE,
    device  = 0,
    augment = True,
    hsv_s   = 0.8,
    hsv_v   = 0.5,
    mosaic  = 1.0,
    flipud  = 0.0,
    fliplr  = 0.5,
    degrees = 5.0,             # Small rotation for varied rider angles
)

s3_best = Path('runs/detect/safety_net/weights/best.pt')
s3_out  = OUTPUT_DIR / 'safety_net.pt'
if s3_best.exists():
    shutil.copy(s3_best, s3_out)
    print(f'\n✅ Stage 3 saved: {s3_out}')
    map50 = s3_results.results_dict.get('metrics/mAP50(B)', 'N/A')
    print(f'   mAP50: {map50}')
else:
    print('⚠️ Safety net best.pt not found')

In [ ]:
# ══════════════════════════════════════════════════════════════
# POTHOLE NET — Independent parallel model (fresh from COCO)
# ══════════════════════════════════════════════════════════════
print('='*60)
print('  POTHOLE NET: Training Road Surface Detector')
print('  Base: yolov8n.pt (COCO) — runs parallel, not in chain')
print('='*60)

ph_model   = YOLO('yolov8n.pt')
ph_results = ph_model.train(
    data    = str(POTHOLE_YAML),
    epochs  = POTHOLE_EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    name    = 'pothole_net',
    patience= PATIENCE,
    device  = 0,
    augment = True,
    hsv_v   = 0.6,   # High brightness variation — potholes look different wet vs dry
    mosaic  = 0.8,
)

ph_best = Path('runs/detect/pothole_net/weights/best.pt')
ph_out  = OUTPUT_DIR / 'pothole_net.pt'
if ph_best.exists():
    shutil.copy(ph_best, ph_out)
    print(f'\n✅ Pothole Net saved: {ph_out}')
    map50 = ph_results.results_dict.get('metrics/mAP50(B)', 'N/A')
    print(f'   mAP50: {map50}')
else:
    print('⚠️ Pothole net best.pt not found')

In [ ]:
# ── FINAL SUMMARY ────────────────────────────────────────────
print('\n' + '='*60)
print('  🏁 VAHAAN 3-STAGE TRAINING COMPLETE')
print('='*60)

model_map = {
    'stage1_vahaan.pt': ('Stage 1 — Vehicle Classifier', s1_results),
    'safety_net.pt':    ('Stage 3 — Safety Violations',  s3_results),
    'pothole_net.pt':   ('Pothole — Road Surface',       ph_results),
}

for fname, (label, res) in model_map.items():
    p = OUTPUT_DIR / fname
    exists = '✅' if p.exists() else '❌'
    map50  = res.results_dict.get('metrics/mAP50(B)', 'N/A')
    map95  = res.results_dict.get('metrics/mAP50-95(B)', 'N/A')
    print(f'  {exists} {label}')
    print(f'     mAP50: {map50}  |  mAP50-95: {map95}')
    print(f'     File:  {p}')
    print()

print('📥 Download from: Kaggle → right panel → Output → vahaan_models/')
print('\n📌 Deployment:')
print('   Drop all .pt files into your VAHAAN project root')
print('   stage1_vahaan.pt  →  replaces best_new.pt')
print('   safety_net.pt     →  auto-detected by SafetyEngine')
print('   pothole_net.pt    →  auto-detected by PotholeAlertSystem')
print('   Restart API → 3-stage chain activates automatically')

In [ ]:
# ── OPTIONAL: Push trained models back to HuggingFace ────────
# Uncomment when ready to version your trained models on HF

# from huggingface_hub import HfApi
# MODEL_REPO = HF_REPO.split('/')[0] + '/vahaan-safety-models'
# api = HfApi()
# api.create_repo(MODEL_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
# for pt in OUTPUT_DIR.glob('*.pt'):
#     api.upload_file(
#         path_or_fileobj = str(pt),
#         path_in_repo    = pt.name,
#         repo_id         = MODEL_REPO,
#         repo_type       = 'model',
#         token           = HF_TOKEN,
#     )
#     print(f'☁️  Uploaded {pt.name} → hf.co/{MODEL_REPO}')